# Balance hídrico por cuenca nivel 2 — exploración en Jupyter

Mismo dato que usa `streamlit_app.py`, pero acá lo podés ir mirando celda a
celda: mapa, ranking de cuencas por volumen, desglose de obras por tipo y
tabla completa.

Corré las celdas en orden (Shift+Enter). Si `data/solicitudes_limpio.csv` y
`data/cuencas_n2.geojson` no existen todavía, corré primero `prepare_data.py`
(o copiá este notebook a la carpeta `streamlit_app/`, que ya los tiene).

In [24]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pydeck as pdk
from IPython.display import HTML, display

# Busca la carpeta data/ sola (funciona corras el notebook desde donde lo
# corras). Si no la encuentra, completá la ruta completa a mano acá abajo.
_candidatas = [Path("data"), Path("streamlit_app/data"), Path.cwd() / "data"]
DATA_DIR = next((c for c in _candidatas if (c / "solicitudes_limpio.csv").exists()), None)
if DATA_DIR is None:
    # <-- si sigue sin encontrarla, descomentá y completá esta línea:
    # DATA_DIR = Path(r"C:\Georgi No Borrar\DINAGUA\Estado del ambiente\streamlit_app\data")
    raise FileNotFoundError(
        "No encontré la carpeta data/. Completá DATA_DIR a mano con la ruta "
        "completa a la carpeta streamlit_app/data en tu computadora."
    )
print("Usando datos de:", DATA_DIR.resolve())

TIPOS_ORDER = [
    "Represa Grande", "Represa Mediana", "Represa Chica",
    "Tajamar Grande", "Tajamar Mediano", "Tajamar Chico",
    "Tanque Excavado", "Reservorio/Otros",
]
N1_COLORS = {
    "Río Uruguay": "#2a78d6",
    "Río de la Plata": "#eb6834",
    "Océano Atlántico": "#1baf7a",
    "Laguna Merín": "#eda100",
    "Río Negro": "#e87ba4",
    "Santa Lucía": "#008300",
}
SEQ_SCALE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

# fondo del mapa de puntos (CartoDB, no necesita token de API)
MAPA_ESTILO = "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json"


def miles(n):
    """1234567 -> '1.234.567' (separador de miles a la uruguaya)."""
    return f"{n:,.0f}".replace(",", ".")

Usando datos de: C:\Georgi No Borrar\DINAGUA\Estado del ambiente\data


## 1. Cargar los datos

In [25]:
df = pd.read_csv(DATA_DIR / "solicitudes_limpio.csv")
df["codcuenca"] = df["codcuenca"].astype(int)

with open(DATA_DIR / "cuencas_n2.geojson", encoding="utf-8") as f:
    geojson = json.load(f)

props = pd.DataFrame([feat["properties"] for feat in geojson["features"]])
props["codcuenca"] = props["codcuenca"].astype(int)
props = props.rename(columns={"nombre_cue": "nombre_cuenca", "area": "area_km2"})
props = props[["codcuenca", "nombre_cuenca", "area_km2", "cabecera"]]

nombre_por_codigo = dict(zip(props["codcuenca"], props["nombre_cuenca"]))
n1_por_codigo = df.groupby("codcuenca")["cuenca_n1"].agg(lambda s: s.mode().iat[0]).to_dict()

print(f"{len(df):,} solicitudes cargadas, en {df['codcuenca'].nunique()} cuencas nivel 2".replace(",", "."))
df.head()

2.067 solicitudes cargadas. en 48 cuencas nivel 2


,cuenca_n1,codcuenca,tipo_obra,volumen,departamento,uso,destino,lat,lon,estado,accion_solicitud,tipo_resolucion,curso,area_cuenca_ha,tipo_obra_agr
0,Río Uruguay,10,Represa Mediana,4097400.0,ARTIGAS,Otros usos agropecuarios,Abrevadero de ganado,-30.551232,-56.907142,Registrada,Nueva,Concesión,Cda. Zanja José Juan,875.0,Represa Mediana
1,Río Uruguay,10,Represa Mediana,2237000.0,ARTIGAS,Otros usos agropecuarios,Abrevadero de ganado,-30.768699,-56.895001,Registrada,Renovación,Concesión,Cda. del Sauce,720.0,Represa Mediana
2,Río Uruguay,10,Represa Chica,0.0,ARTIGAS,Riego,Arroz,-30.539455,-56.603028,Registrada,Renovación,Concesión,Cda. s/n,80.0,Represa Chica
3,Río Uruguay,10,Represa Mediana,2000000.0,ARTIGAS,Riego,Arroz,-30.378384,-57.398262,Registrada,Nueva,Concesión,Cda. Del Sauzal I,500.0,Represa Mediana
4,Río Uruguay,10,Represa Chica,648000.0,ARTIGAS,Riego,Arroz,-30.745597,-56.852971,Registrada,Nueva,Concesión,Cda. / Ao. Cuaro,150.0,Represa Chica


## 2. KPIs generales

In [26]:
print(f"Volumen total:        {df['volumen'].sum() / 1e6:,.1f} hm³".replace(",", "."))
print(f"Obras registradas:    {len(df):,}".replace(",", "."))
print(f"Cuencas nivel 2:      {df['codcuenca'].nunique()}")
print(f"Volumen medio / obra: {df['volumen'].mean():,.0f} m³".replace(",", "."))

Volumen total:        1.292.9 hm³
Obras registradas:    2.067
Cuencas nivel 2:      48
Volumen medio / obra: 625.481 m³


## 3. Agregaciones: volumen y obras por cuenca nivel 2

In [27]:
por_cuenca = (
    df.groupby("codcuenca")
    .agg(volumen=("volumen", "sum"), n_obras=("volumen", "count"))
    .reset_index()
)
por_cuenca["nombre_cuenca"] = por_cuenca["codcuenca"].map(nombre_por_codigo)
por_cuenca["cuenca_n1"] = por_cuenca["codcuenca"].map(n1_por_codigo)
por_cuenca["area_km2"] = por_cuenca["codcuenca"].map(dict(zip(props["codcuenca"], props["area_km2"])))
por_cuenca = por_cuenca.sort_values("volumen", ascending=False)

tipo_por_cuenca = (
    df.pivot_table(index="codcuenca", columns="tipo_obra_agr", values="volumen", aggfunc="count", fill_value=0)
    .reindex(columns=TIPOS_ORDER, fill_value=0)
)

por_cuenca.head(10)

,codcuenca,volumen,n_obras,nombre_cuenca,cuenca_n1,area_km2
30,50,1.677591e+08,90,RÍO NEGRO entre nacientes y Río Tacuarembó,Río Negro,11398.0
25,42,1.380000e+08,5,LAGUNA MERÍN entre Río Tacuarí y Río Cebollatí,Laguna Merín,1222.0
27,44,9.755829e+07,61,RÍO CEBOLLATÍ,Laguna Merín,12117.0
0,10,9.022958e+07,42,RÍO CUAREIM,Río Uruguay,8222.0
33,53,7.398535e+07,55,RÍO TACUAREMBÓ entre Ao. Tacuarembó Chico y Rí...,Río Negro,5974.0
38,58,6.680708e+07,73,RÍO NEGRO entre Rincón de Palmar y Río Uruguay,Río Negro,8647.0
3,13,6.546317e+07,30,RÍO ARAPEY GRANDE,Río Uruguay,9698.0
24,41,6.282415e+07,40,RÍO TACUARÍ,Laguna Merín,4684.0
1,11,6.205323e+07,35,RÍO URUGUAY entre Río Cuareim y Río Arapey Grande,Río Uruguay,2583.0
36,56,4.803496e+07,79,RÍO YÍ,Río Negro,13720.0


## 4. Mapa — volumen concedido por cuenca

In [28]:
map_df = por_cuenca[por_cuenca["codcuenca"].isin([f["properties"]["codcuenca"] for f in geojson["features"]])]

lons = [pt[0] for feat in geojson["features"] for ring in feat["geometry"]["coordinates"]
        for poly in (ring if feat["geometry"]["type"] == "MultiPolygon" else [ring]) for pt in poly]
lats = [pt[1] for feat in geojson["features"] for ring in feat["geometry"]["coordinates"]
        for poly in (ring if feat["geometry"]["type"] == "MultiPolygon" else [ring]) for pt in poly]
center = {"lat": (min(lats) + max(lats)) / 2, "lon": (min(lons) + max(lons)) / 2}

fig_map = px.choropleth_map(
    map_df, geojson=geojson, locations="codcuenca", featureidkey="properties.codcuenca",
    color="volumen", color_continuous_scale=SEQ_SCALE, hover_name="nombre_cuenca",
    hover_data={"codcuenca": True, "volumen": ":,.0f", "n_obras": True},
    map_style="white-bg", center=center, zoom=5.05, opacity=0.9,
)
fig_map.update_layout(margin=dict(l=0, r=0, t=0, b=0), height=560, coloraxis_colorbar=dict(title="Volumen (m³)"))
fig_map

## 5. Ranking de cuencas nivel 2, agrupadas por su cuenca nivel 1

Las 48 cuencas nivel 2 (no solo un top), un panel por cada una de las 6 cuencas nivel 1, cada panel ordenado por volumen.

In [29]:
plot_df = por_cuenca.copy()
plot_df["label"] = plot_df["nombre_cuenca"] + " (#" + plot_df["codcuenca"].astype(str) + ")"
plot_df = plot_df.sort_values(["cuenca_n1", "volumen"])

fig_rank = px.bar(
    plot_df, x="volumen", y="label", orientation="h",
    color="cuenca_n1", color_discrete_map=N1_COLORS,
    facet_row="cuenca_n1",
    category_orders={"cuenca_n1": list(N1_COLORS.keys())},
    labels={"volumen": "Volumen (m³)", "label": "", "cuenca_n1": "Cuenca nivel 1"},
)
fig_rank.update_yaxes(matches=None, showticklabels=True)
fig_rank.update_xaxes(matches="x")
fig_rank.update_layout(height=1500, margin=dict(l=0, r=0, t=30, b=0), showlegend=False)
fig_rank.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_rank

## 6. Obras por tipo — para una cuenca puntual

Cambiá `CODCUENCA` por el código que quieras mirar (los códigos están en
`por_cuenca["codcuenca"]`, celda 3).

In [30]:
CODCUENCA = 50  # <-- cambiá este código

tipo_totales = tipo_por_cuenca.loc[CODCUENCA].reindex(TIPOS_ORDER, fill_value=0)
nombre = nombre_por_codigo.get(CODCUENCA, f"#{CODCUENCA}")

fig_tipo = go.Figure(go.Bar(
    x=tipo_totales.values, y=tipo_totales.index, orientation="h",
    marker_color="#2a78d6", text=tipo_totales.values, textposition="outside",
))
fig_tipo.update_layout(
    title=f"{nombre} (#{CODCUENCA})", height=380,
    xaxis_title="Cantidad de obras", yaxis=dict(autorange="reversed"),
)
fig_tipo

## 7. Resumen por cuenca nivel 1

In [31]:
resumen_n1 = (
    df.groupby("cuenca_n1")
    .agg(volumen=("volumen", "sum"), obras=("volumen", "count"), cuencas=("codcuenca", "nunique"))
    .reset_index()
    .sort_values("volumen")
)
fig_n1 = px.bar(
    resumen_n1, x="volumen", y="cuenca_n1", orientation="h", color="cuenca_n1",
    color_discrete_map=N1_COLORS, text=resumen_n1["obras"].astype(str) + " obras",
    labels={"volumen": "Volumen (m³)", "cuenca_n1": ""},
)
fig_n1.update_traces(textposition="outside")
fig_n1.update_layout(height=380, showlegend=False)
fig_n1

## 8. Tabla completa

In [32]:
tabla = por_cuenca[["codcuenca", "nombre_cuenca", "cuenca_n1", "area_km2", "volumen", "n_obras"]].reset_index(drop=True)
tabla.columns = ["Código", "Cuenca nivel 2", "Cuenca nivel 1", "Área (km²)", "Volumen (m³)", "Obras"]
tabla

,Código,Cuenca nivel 2,Cuenca nivel 1,Área (km²),Volumen (m³),Obras
0,50,RÍO NEGRO entre nacientes y Río Tacuarembó,Río Negro,11398.0,1.677591e+08,90
1,42,LAGUNA MERÍN entre Río Tacuarí y Río Cebollatí,Laguna Merín,1222.0,1.380000e+08,5
2,44,RÍO CEBOLLATÍ,Laguna Merín,12117.0,9.755829e+07,61
3,10,RÍO CUAREIM,Río Uruguay,8222.0,9.022958e+07,42
4,53,RÍO TACUAREMBÓ entre Ao. Tacuarembó Chico y Rí...,Río Negro,5974.0,7.398535e+07,55
5,58,RÍO NEGRO entre Rincón de Palmar y Río Uruguay,Río Negro,8647.0,6.680708e+07,73
6,13,RÍO ARAPEY GRANDE,Río Uruguay,9698.0,6.546317e+07,30
7,41,RÍO TACUARÍ,Laguna Merín,4684.0,6.282415e+07,40
8,11,RÍO URUGUAY entre Río Cuareim y Río Arapey Grande,Río Uruguay,2583.0,6.205323e+07,35
9,56,RÍO YÍ,Río Negro,13720.0,4.803496e+07,79


## 9. Distribución geográfica de las solicitudes (mapa de puntos)

Un punto por solicitud/obra, ubicado según su Latitud/Longitud. Primero
armamos `df_mapa` (filtrado a las coordenadas dentro de Uruguay), y después
dos versiones: coloreado por tipo de uso, y coloreado + tamaño por volumen.

In [33]:
df_mapa = df[["lat", "lon", "uso", "volumen"]].copy()
df_mapa["lat"] = pd.to_numeric(df_mapa["lat"], errors="coerce")
df_mapa["lon"] = pd.to_numeric(df_mapa["lon"], errors="coerce")
df_mapa = df_mapa[
    df_mapa["lat"].between(-35.5, -30.0) & df_mapa["lon"].between(-59.5, -53.0)
].dropna(subset=["lat", "lon"])

print(f"{miles(len(df_mapa))} de {miles(len(df))} solicitudes quedaron georreferenciadas")
df_mapa.head()

2.065 de 2.067 solicitudes quedaron georreferenciadas


,lat,lon,uso,volumen
0,-30.551232,-56.907142,Otros usos agropecuarios,4097400.0
1,-30.768699,-56.895001,Otros usos agropecuarios,2237000.0
2,-30.539455,-56.603028,Riego,0.0
3,-30.378384,-57.398262,Riego,2000000.0
4,-30.745597,-56.852971,Riego,648000.0


### 9.1 Coloreado por tipo de uso

In [34]:
usos_unicos = sorted(df_mapa["uso"].dropna().unique())
tab10 = plt.colormaps["tab10"].resampled(max(len(usos_unicos), 1))
color_map = {}
for i, uso in enumerate(usos_unicos):
    r, g, b, _ = tab10(i)
    color_map[uso] = [int(r * 255), int(g * 255), int(b * 255), 200]
df_mapa["color"] = df_mapa["uso"].apply(lambda u: color_map.get(u, [150, 150, 150, 180]))

leyenda_html = "<div style='display:flex;flex-wrap:wrap;gap:8px;margin-bottom:12px;'>"
for i, uso in enumerate(usos_unicos):
    r, g, b, _ = tab10(i)
    hex_c = "#{:02x}{:02x}{:02x}".format(int(r * 255), int(g * 255), int(b * 255))
    leyenda_html += (
        f"<span style='background:{hex_c};color:white;"
        f"padding:3px 10px;border-radius:12px;font-size:12px;'>{uso}</span>"
    )
leyenda_html += "</div>"
display(HTML(leyenda_html))
print(f"📍 {miles(len(df_mapa))} registros georreferenciados")

layer_uso = pdk.Layer(
    "ScatterplotLayer", data=df_mapa,
    get_position=["lon", "lat"], get_color="color",
    get_radius=3000, pickable=True, auto_highlight=True,
)
pdk.Deck(
    layers=[layer_uso],
    initial_view_state=pdk.ViewState(latitude=-32.5, longitude=-56.0, zoom=6),
    map_style=MAPA_ESTILO,
    tooltip={"text": "Uso: {uso}"},
)

📍 2.065 registros georreferenciados


{
  "initialViewState": {
    "latitude": -32.5,
    "longitude": -56.0,
    "zoom": 6
  },
  "layers": [
    {
      "@@type": "ScatterplotLayer",
      "autoHighlight": true,
      "data": [
        {
          "color": [
            227,
            119,
            194,
            200
          ],
          "lat": -30.551231870022185,
          "lon": -56.90714163945258,
          "uso": "Otros usos agropecuarios",
          "volumen": 4097400.0
        },
        {
          "color": [
            227,
            119,
            194,
            200
          ],
          "lat": -30.768699089046773,
          "lon": -56.89500094320569,
          "uso": "Otros usos agropecuarios",
          "volumen": 2237000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.53945455710129,
          "lon": -56.60302815289879,
          "uso": "Riego",
          "volumen": 0.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.378383672377165,
          "lon": -57.39826217982998,
          "uso": "Riego",
          "volumen": 2000000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.74559722545752,
          "lon": -56.85297064221824,
          "uso": "Riego",
          "volumen": 648000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.262517684946168,
          "lon": -56.91739615577281,
          "uso": "Riego",
          "volumen": 7082000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.90636341990602,
          "lon": -56.563942052570965,
          "uso": "Riego",
          "volumen": 2963000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.32016356950844,
          "lon": -56.75689879211755,
          "uso": "Riego",
          "volumen": 2753000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.23752531450046,
          "lon": -56.64078666348033,
          "uso": "Riego",
          "volumen": 2801000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.27031675760388,
          "lon": -56.846822212270766,
          "uso": "Riego",
          "volumen": 478000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.30232853007597,
          "lon": -56.789989726047615,
          "uso": "Riego",
          "volumen": 2979000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.145210731465752,
          "lon": -57.02400513579964,
          "uso": "Riego",
          "volumen": 1690000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.75821,
          "lon": -56.67654,
          "uso": "Riego",
          "volumen": 495000.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.76647,
          "lon": -56.67929,
          "uso": "Riego",
          "volumen": 145550.0
        },
        {
          "color": [
            188,
            189,
            34,
            200
          ],
          "lat": -30.68331,
          "lon": -56.33579,
          "uso": "Riego",
          "volumen": 354200.0
        },
        {
          "color": [
            1

### 9.2 Coloreado y proporcional al volumen

In [35]:
df_mapa_vol = df_mapa.dropna(subset=["volumen"]).copy()
vol_p95 = df_mapa_vol["volumen"].quantile(0.95) or 1
df_mapa_vol["vol_norm"] = (df_mapa_vol["volumen"].clip(upper=vol_p95) / vol_p95).fillna(0)
cmap_vol = plt.colormaps["coolwarm"]
df_mapa_vol["color"] = df_mapa_vol["vol_norm"].apply(
    lambda n: [int(c * 255) for c in cmap_vol(n)[:3]] + [200]
)
df_mapa_vol["radio"] = (1500 + df_mapa_vol["vol_norm"] * 6500).astype(int)

vol_min = int(df_mapa_vol["volumen"].min())
vol_med = int(df_mapa_vol["volumen"].median())
vol_max = int(vol_p95)
r_b, g_b, b_b, _ = cmap_vol(0.0)
r_m, g_m, b_m, _ = cmap_vol(0.5)
r_a, g_a, b_a, _ = cmap_vol(1.0)
hex_b = "#{:02x}{:02x}{:02x}".format(int(r_b * 255), int(g_b * 255), int(b_b * 255))
hex_m = "#{:02x}{:02x}{:02x}".format(int(r_m * 255), int(g_m * 255), int(b_m * 255))
hex_a = "#{:02x}{:02x}{:02x}".format(int(r_a * 255), int(g_a * 255), int(b_a * 255))

display(HTML(
    f"<div style='display:flex;align-items:center;gap:12px;"
    f"margin-bottom:12px;font-size:12px;'>"
    f"<span>Volumen:</span>"
    f"<span style='background:{hex_b};color:white;padding:3px 10px;"
    f"border-radius:12px;'>Bajo (&lt;{miles(vol_min)} m³)</span>"
    f"<span style='background:{hex_m};color:white;padding:3px 10px;"
    f"border-radius:12px;'>Medio (~{miles(vol_med)} m³)</span>"
    f"<span style='background:{hex_a};color:white;padding:3px 10px;"
    f"border-radius:12px;'>Alto (&gt;{miles(vol_max)} m³)</span>"
    f"<span style='color:#aaa;'>Tamaño proporcional al volumen</span>"
    f"</div>"
))
print(f"📍 {miles(len(df_mapa_vol))} registros con volumen georreferenciados")

layer_vol = pdk.Layer(
    "ScatterplotLayer", data=df_mapa_vol,
    get_position=["lon", "lat"], get_color="color",
    get_radius="radio", pickable=True, auto_highlight=True,
)
pdk.Deck(
    layers=[layer_vol],
    initial_view_state=pdk.ViewState(latitude=-32.5, longitude=-56.0, zoom=6),
    map_style=MAPA_ESTILO,
    tooltip={"text": "Volumen: {volumen} m³\nUso: {uso}"},
)

📍 2.065 registros con volumen georreferenciados


{
  "initialViewState": {
    "latitude": -32.5,
    "longitude": -56.0,
    "zoom": 6
  },
  "layers": [
    {
      "@@type": "ScatterplotLayer",
      "autoHighlight": true,
      "data": [
        {
          "color": [
            179,
            3,
            38,
            200
          ],
          "lat": -30.551231870022185,
          "lon": -56.90714163945258,
          "radio": 8000,
          "uso": "Otros usos agropecuarios",
          "vol_norm": 1.0,
          "volumen": 4097400.0
        },
        {
          "color": [
            242,
            145,
            115,
            200
          ],
          "lat": -30.768699089046773,
          "lon": -56.89500094320569,
          "radio": 6495,
          "uso": "Otros usos agropecuarios",
          "vol_norm": 0.7685486661025378,
          "volumen": 2237000.0
        },
        {
          "color": [
            58,
            76,
            192,
            200
          ],
          "lat": -30.53945455710129,
          "lon": -56.60302815289879,
          "radio": 1500,
          "uso": "Riego",
          "vol_norm": 0.0,
          "volumen": 0.0
        },
        {
          "color": [
            247,
            177,
            148,
            200
          ],
          "lat": -30.378383672377165,
          "lon": -57.39826217982998,
          "radio": 5966,
          "uso": "Riego",
          "vol_norm": 0.687124422085416,
          "volumen": 2000000.0
        },
        {
          "color": [
            130,
            165,
            251,
            200
          ],
          "lat": -30.74559722545752,
          "lon": -56.85297064221824,
          "radio": 2947,
          "uso": "Riego",
          "vol_norm": 0.22262831275567477,
          "volumen": 648000.0
        },
        {
          "color": [
            179,
            3,
            38,
            200
          ],
          "lat": -30.262517684946168,
          "lon": -56.91739615577281,
          "radio": 8000,
          "uso": "Riego",
          "vol_norm": 1.0,
          "volumen": 7082000.0
        },
        {
          "color": [
            179,
            3,
            38,
            200
          ],
          "lat": -30.90636341990602,
          "lon": -56.563942052570965,
          "radio": 8000,
          "uso": "Riego",
          "vol_norm": 1.0,
          "volumen": 2963000.0
        },
        {
          "color": [
            198,
            53,
            52,
            200
          ],
          "lat": -30.32016356950844,
          "lon": -56.75689879211755,
          "radio": 7647,
          "uso": "Riego",
          "vol_norm": 0.9458267670005751,
          "volumen": 2753000.0
        },
        {
          "color": [
            193,
            42,
            48,
            200
          ],
          "lat": -30.23752531450046,
          "lon": -56.64078666348033,
          "radio": 7755,
          "uso": "Riego",
          "vol_norm": 0.9623177531306251,
          "volumen": 2801000.0
        },
        {
          "color": [
            111,
            145,
            242,
            200
          ],
          "lat": -30.27031675760388,
          "lon": -56.846822212270766,
          "radio": 2567,
          "uso": "Riego",
          "vol_norm": 0.1642227368784144,
          "volumen": 478000.0
        },
        {
          "color": [
            179,
            3,
            38,
            200
          ],
          "lat": -30.30232853007597,
          "lon": -56.789989726047615,
          "radio": 8000,
          "uso": "Riego",
          "vol_norm": 1.0,
          "volumen": 2979000.0
        },
        {
          "color": [
            238,
            207,
            190,
            200
          ],
          "lat": -30.145210731465752,
          "lon": -57.02400513579964,
          "radio": 5274,
          "uso": "Riego",
          "vol_norm": 0.5806201366621765,
          "volumen": 1690000.0
        },
        {
          "c